# प्राधान्य सदस्य मिडलवेयरसह हॉटेल बुकिंग

हा नोटबुक Microsoft Agent Framework वापरून **फंक्शन-आधारित मिडलवेयर** दाखवतो. आम्ही सशर्त वर्कफ्लो उदाहरणावर आधारित आहोत आणि एक मिडलवेयर स्तर जोडतो जो प्राधान्य सदस्यांना विशेष अधिकार देतो.

## तुम्हाला काय शिकायला मिळेल:
1. **फंक्शन-आधारित मिडलवेयर**: फंक्शन परिणामांमध्ये अडथळा आणा आणि बदल करा
2. **कॉन्टेक्स्ट ऍक्सेस**: कार्यान्वयनानंतर `context.result` वाचा आणि बदला
3. **बिझनेस लॉजिक अंमलबजावणी**: प्राधान्य सदस्य लाभ
4. **परिणाम ओव्हरराईड**: वापरकर्ता स्थितीवर आधारित फंक्शन परिणाम बदला
5. **समान वर्कफ्लो, वेगळे परिणाम**: मिडलवेयर-प्रेरित वर्तन बदल

## मिडलवेयरसह वर्कफ्लो आर्किटेक्चर:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## सशर्त वर्कफ्लोपासून प्रमुख फरक:

**मिडलवेयरशिवाय** (14-conditional-workflow.ipynb):
- पॅरिसमध्ये खोल्या नाहीत → alternative_agent कडे मार्गदर्शित करा

**मिडलवेयरसह** (हा नोटबुक):
- नियमित वापरकर्ता + पॅरिस → खोल्या नाहीत → alternative_agent कडे मार्गदर्शित करा
- प्राधान्य वापरकर्ता + पॅरिस → 🌟 मिडलवेयर ओव्हरराईड करतो! → उपलब्ध → booking_agent कडे मार्गदर्शित करा

## आवश्यक अटी:
- Microsoft Agent Framework स्थापित
- सशर्त वर्कफ्लोची समज (14-conditional-workflow.ipynb पहा)
- GitHub टोकन किंवा OpenAI API की
- मिडलवेयर पॅटर्नचे मूलभूत ज्ञान


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Step 1: संरचित आउटपुटसाठी पायडँटिक मॉडेल्स परिभाषित करा

हे मॉडेल्स एजंट परत करणार्‍या **योजनेची** व्याख्या करतात. आम्ही `priority_override` क्षेत्र जोडले आहे जेव्हा मिडलवेयर उपलब्धता परिणाम बदलतो तेव्हा त्याचा ट्रॅक ठेवण्यासाठी.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: प्राधान्य सदस्य डेटाबेस परिभाषित करा

या डेमोसाठी, आपण एक प्राधान्य सदस्यत्व डेटाबेस अनुकरण करू. उत्पादनात, हे खऱ्या डेटाबेस किंवा API ला क्वेरी करील.

**प्राधान्य सदस्य:**
- `alice@example.com` - व्हीआयपी सदस्य
- `bob@example.com` - प्रीमियम सदस्य  
- `priority_user` - चाचणी खाते


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## चरण 3: हॉटेल बुकिंग टूल तयार करा

सशर्त वर्कफ्लो प्रमाणेच, परंतु आता ते मिडलवेअरने अडवले जाईल!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 4: 🌟 प्राधान्य तपासणी मिडलवेअर तयार करा (महत्त्वाचे वैशिष्ट्य!)

हा या नोटबुकची **मूलभूत कार्यक्षमता** आहे. मिडलवेअर:

1. hotel_booking फंक्शन कॉल **थांबवते**
2. `next(context)` कॉल करून फंक्शन सामान्यपणे **कार्यान्वित करते**
3. `context.result` मधील निकाल **तपासते**
4. वापरकर्ता प्राधान्ययुक्त असल्यास आणि खोल्या उपलब्ध नसल्यास निकाल **ओवरराइड करते**
5. सुधारित निकाल एजंटकडे **परत करते**

**महत्त्वाचा नमुना:**
```python
async def my_middleware(context, next):
    await next(context)  # फंक्शन कार्यान्वित करा
    # आता context.result मध्ये फंक्शनचं आउटपुट आहे
    if some_condition:
        context.result = new_value  # अधिलेखित करा!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## पाऊल 5: राऊटिंगसाठी अटी फंक्शन्स Define करा

स्ट्रक्चरल आउटपुट तपासण्यासाठी त्याच अटी फंक्शन्स conditional workflow प्रमाणे - ज्या राऊटिंग ठरवतात.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## टप्पा ६: कस्टम डिस्प्ले एक्झिक्युटर तयार करा

स्वत:चा एक्झिक्युटर जसा आधी होता तसाच - अंतिम वर्कफ्लो आउटपुट दाखवतो.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## टप्पा 7: पर्यावरणीय चल परिवर्तने लोड करा

LLM क्लायंट (GitHub मॉडेल्स किंवा OpenAI) कॉन्फिगर करा.


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## टप्पा 8: मिडलवेअरसह AI एजंट तयार करा

**मुख्य फरक:** availability_agent तयार करताना, आपण `middleware` पॅरामीटर पास करतो!

असे आपण priority_check_middleware एजंटच्या फंक्शन कॉल पाईपलाइनमध्ये इंजेक्ट करतो.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 9: वर्कफ्लो तयार करा

पूर्वीप्रमाणेच वर्कफ्लो रचना - उपलब्धतेनुसार सशर्त मार्गदर्शने.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## पाऊल 10: चाचणी प्रकरण 1 - पॅरिसमधील नियमित वापरकर्ता (ओव्हरराईड नाही)

एक नियमित वापरकर्ता पॅरिस बुक करण्याचा प्रयत्न करतो → खोल्या नाहीत → alternative_agent कडे मार्गदर्शित केले जाते


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 पॅरिसमधील प्राधान्य वापरकर्ता (ओव्हरराईडसह!)

एक प्राधान्य सदस्य पॅरिससाठी बुक करण्याचा प्रयत्न करतो → सुरुवातीला उपलब्ध खोल्या नाहीत → 🌟 मिडलवेअरने ओव्हरराईड केले! → booking_agent कडे मार्गक्रमण

**हा मिडलवेअर शक्तीचा मुख्य प्रदर्शन आहे!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Step 12: चाचणी प्रकरण 3 - स्टॉकहोममधील प्राधान्य वापरकर्ता (आधीच उपलब्ध)

प्राधान्य वापरकर्ता स्टॉकहोमचा प्रयत्न करतो → खोल्या उपलब्ध → कोणतीही ओव्हरराईड आवश्यक नाही → booking_agent कडे मार्गदर्शन

हे दर्शवते की मिडलवेअर फक्त आवश्यक तेव्हा काम करते!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## मुख्य मुद्दे आणि मिडलवेअर संकल्पना

### ✅ आपण काय शिकलात:

#### **1. फंक्शन-आधारित मिडलवेअर पॅटर्न**

मिडलवेअर साध्या async फंक्शनचा वापर करून फंक्शन कॉल्सना इंटरसेप्ट करते:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # फंक्शन अंमलबजावणीपूर्वी
    print("Intercepting...")
    
    # फंक्शन अंमलबजावणी करा
    await next(context)
    
    # फंक्शन अंमलबजावणीनंतर - निकाल तपासा
    if context.result:
        # आवश्यक असल्यास निकाल सुधारित करा
        context.result = modified_value
```

#### **2. कॉन्टेक्स्ट ऍक्सेस आणि परिणाम ओव्हरराईड**

- `context.function` - कॉल होणा-या फंक्शनमध्ये ऍक्सेस करा
- `context.arguments` - फंक्शनच्या आर्ग्युमेंट्स वाचा
- `context.kwargs` - अतिरिक्त पॅरामिटर्समध्ये ऍक्सेस करा
- `await next(context)` - फंक्शन चालवा
- `context.result` - फंक्शनचा आउटपुट वाचा/संपादित करा

#### **3. बिझनेस लॉजिक अंमलबजावणी**

आमचे मिडलवेअर प्राधान्य सदस्य फायदे अंमलात आणते:
- **नियमित वापरकर्ते**: कोणतीही बदल नाही, मानक कार्यप्रणाली
- **प्राथमिकता वापरकर्ते**: "कोणतीही उपलब्धता नाही" → "उपलब्ध" ओव्हरराईड करा
- **अटींचे लॉजिक**: केवळ गरज असताना ओव्हरराईड करते

#### **4. सारखी कार्यप्रणाली, वेगळे परिणाम**

मिडलवेअरची शक्ति:
- ✅ कार्यप्रणालीच्या रचनेत कोणताही बदल नाही
- ✅ टूल फंक्शनमध्ये कोणताही बदल नाही
- ✅ अटींच्या मार्गदर्शन लॉजिकमध्ये कोणताही बदल नाही
- ✅ फक्त मिडलवेअर → वेगळे वर्तन!

### 🚀 वास्तविक जगातील वापर:

1. **VIP/प्रीमियम वैशिष्ट्ये**
   - प्रिमियम वापरकर्त्यांसाठी रेट लिमिट्स ओव्हरराईड करा
   - संसाधनांवर प्राथमिकता ऍक्सेस द्या
   - प्रिमियम वैशिष्ट्ये डायनॅमिकली अनलॉक करा

2. **A/B चाचणी**
   - वापरकर्त्यांना वेगवेगळ्या अंमलबजावणीकडे मार्गदर्शन करा
   - विशिष्ट वापरकर्त्यांसह नवीन वैशिष्ट्यांचे चाचणी करा
   - हळूहळू फीचर रोलआउट करा

3. **सुरक्षा आणि पालन**
   - फंक्शन कॉल्सचे ऑडिट करा
   - संवेदनशील ऑपरेशन्स ब्लॉक करा
   - बिझनेस नियमांची अंमलबजावणी करा

4. **कार्यक्षमता ऑप्टिमायझेशन**
   - विशिष्ट वापरकर्त्यांसाठी निकाल कॅश करा
   - शक्य असल्यास महागड्या ऑपरेशन्स वगळा
   - डायनॅमिक रिसोर्स अलोकेशन

5. **त्रुटी हाताळणी आणि पुनःप्रयास**
   - त्रुटी पकडा आणि मृदुभावाने हाताळा
   - पुनःप्रयास लॉजिक अंमलात आणा
   - पर्यायी अंमलबजावणींकडे फेरवाटा करा

6. **लॉगिंग आणि मॉनिटरिंग**
   - फंक्शन एक्झिक्युशन वेळ ट्रॅक करा
   - पॅरामिटर्स आणि निकाल लॉग करा
   - वापर नमुने मॉनिटर करा

### 🔑 डेकोरेटर्सपासून मुख्य फरक:

| वैशिष्ट्य | डेकोरेटर | मिडलवेअर |
|---------|-----------|------------|
| **स्कोप** | एकच फंक्शन | एजंटमधील सर्व फंक्शन्स |
| **लवचिकता** | परिभाषेत निश्चित | रनटाइमवर डायनॅमिक |
| **कॉन्टेक्स्ट** | मर्यादित | पूर्ण एजंट कॉन्टेक्स्ट |
| **संयोजन** | अनेक डेकोरेटर्स | मिडलवेअर पाईपलाइन |
| **एजंट-आवेयर** | नाही | होय (एजंट स्टेटमध्ये ऍक्सेस) |

### 📚 कधी मिडलवेअर वापरावे:

✅ **मिडलवेअर वापरा जेव्हा:**
- वापरकर्ता/सेशन स्थितीवर आधारित वर्तन बदलायचे असेल
- तुम्हाला अनेक फंक्शन्सवर लॉजिक लागू करायचे असेल
- एजंट लेव्हल कॉन्टेक्स्टमध्ये ऍक्सेस हवा असेल
- क्रॉस-कटिंग चिंता (लॉगिंग, ऑथ, इ.) अंमलात आणत असाल

❌ **मिडलवेअर वापरू नका जेव्हा:**
- सोपी इनपुट वैलिडेशन करायची असेल (Pydantic वापरा)
- फंक्शन-विशिष्ट लॉजिक (फंक्शनमध्येच ठेवा)
- एकदाच होणारे बदल (फक्त फंक्शन बदला)

### 🎓 प्रगत पॅटर्न्स:

```python
# अनेक मिडलवेयर (अमलबजावणी क्रम महत्वाचा!)
middleware=[
    logging_middleware,      # प्रथम लॉग्स
    auth_middleware,         # नंतर ऑथ तपासतो
    cache_middleware,        # नंतर कॅश तपासतो
    rate_limit_middleware,   # नंतर दर मर्यादा लावतो
    priority_check_middleware  # शेवटी प्राधान्य तपासणी
]

# सशर्त मिडलवेयर अमलबजावणी
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # निकाल बदलणे
    else:
        # पूर्णपणे अमलबजावणी वगळा
        context.result = cached_value
```

### 🔗 संबंधित संकल्पना:

- **एजंट मिडलवेअर**: agent.run() कॉल्सना इंटरसेप्ट करते
- **फंक्शन मिडलवेअर**: टूल फंक्शन कॉल्सना इंटरसेप्ट करते (जे आपण वापरले!)
- **मिडलवेअर पाईपलाइन**: क्रमाने चालणारी मिडलवेअरची साखळी
- **कॉन्टेक्स्ट प्रॉपगेशन**: मिडलवेअर साखळीमध्ये स्टेट पास करणे


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
